# Gap study: why flow5 and AeroSandbox differ

Notebook 02 shows the gap. This notebook finds its causes. Each step tests one cause and prints the numbers that
`AeroSandbox/docs/methods_report.md` quotes.

Point of the study: the flow5 T7 point (its speed, its mass), moments about the flow5 MRP (the T7 CoG), elevator 0.
The model now has the tail boom (`tail_boom`, a CREO spoke tube under 1.2.2.4, placeholder).

| Method | Section data | Wing-on-tail downwash | Bodies |
|---|---|---|---|
| flow5 (DAVE-ML, T7) | inviscid panels | yes | fuselage, no boom |
| VLM (`asb.VortexLatticeMethod`) | thin camber line, 2 pi | yes | none |
| LiftingLine (`asb.LiftingLine`) | NeuralFoil | yes | AeroBuildup bodies |
| NonlinearLiftingLine | NeuralFoil | yes | none |
| AeroBuildup (`asb.AeroBuildup`) | NeuralFoil | **no** | Jorgensen bodies |

`RUN_NLL = False` skips the NonlinearLiftingLine rows (about 5 minutes).

In [ ]:
import sys, math, warnings
sys.path.insert(0, "..")                    # AeroSandbox/lab.py
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import aerosandbox as asb
from IPython.display import display
import lab

RUN_NLL = True
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
L = lab.load("currawong")                   # reads aircraft/currawong (read only)
lab.style()
mrp, c_ref, s_ref, b_ref = L.ref["xyz_ref"], L.ref["c_m"], L.ref["s_m2"], L.ref["b_m"]
t7 = lab.t7_reference(L)
K = L.flow5_const()
cond = lab.condition(t7["V"], t7["mass_kg"] * lab.G0, s_ref, c_ref)
cg_now = lab.rollup(L.masses)["cg"]
print("bodies in the model:", [f.name for f in L.airplane.fuselages])
print(f"T7 point: V {t7['V']} m/s, mass {t7['mass_kg']} kg, alpha {t7['alpha_deg']} deg, Re(c) {cond['Re_c']:.3g}")
VLM = dict(spanwise_resolution=16, chordwise_resolution=12)
WT = ["wing_main", "htail"]

## Step 1. Method matrix

Slopes from alpha 1 and 3 deg. NP = neutral point, SM = static margin about the MRP (fraction of c). The VLM rows
show the mesh convergence and the wake direction.

In [ ]:
def row(plane, method, **kw):
    s = lab.slopes(plane, cond, method, **kw)
    s["CL_T7"] = lab.run_aero(plane, cond, t7["alpha_deg"], method, **kw)["CL"]
    return s

rows = {"flow5 (DAVE-ML, T7)": dict(CLa=K["CLa"], Cma=K["Cma"], SM=-K["Cma"] / K["CLa"], NP_x=mrp[0] - K["Cma"] / K["CLa"] * c_ref,
                                    CL0=K["CL0"], alpha_L0_deg=-np.degrees(K["CL0"] / K["CLa"]), CL_T7=t7["CL"])}
for n, (sp, ch) in {"VLM 8x6": (8, 6), "VLM 16x12": (16, 12), "VLM 24x16": (24, 16)}.items():
    rows[n] = row(lab.subset(L, WT, xyz_ref=mrp), "vlm", spanwise_resolution=sp, chordwise_resolution=ch)
rows["VLM 16x12, wake along the wind"] = row(lab.subset(L, WT, xyz_ref=mrp), "vlm", align_trailing_vortices_with_wind=True, **VLM)
rows["LiftingLine"] = row(lab.subset(L, WT, xyz_ref=mrp), "ll")
if RUN_NLL:
    rows["NonlinearLiftingLine"] = row(lab.subset(L, WT, xyz_ref=mrp), "nll")
rows["AeroBuildup wing + tail"] = row(lab.subset(L, WT, xyz_ref=mrp), "buildup")
rows["AeroBuildup full, no boom"] = row(lab.subset(L, WT + ["fin"], xyz_ref=mrp, bodies=["fuselage"]), "buildup")
rows["AeroBuildup full, with boom"] = row(lab.subset(L, WT + ["fin"], True, mrp), "buildup")
M = pd.DataFrame(rows).T
display(M[["CLa", "Cma", "SM", "NP_x", "CL0", "CL_T7"]])

## Step 2. Downwash at the tail

The flow angle that the wing alone makes at the tail (x 1.48 m, z -0.05 m, eight stations across the tail span),
at alpha 1 and 3 deg. AeroBuildup has no such term: it runs each wing alone.

Hand estimate, USAF DATCOM section 4.4.1 (the form in Roskam, Airplane Design Part VI):
`de/da = 4.44 [K_A K_lambda K_H sqrt(cos Lambda_c/4)]^1.19`, `K_A = 1/A - 1/(1 + A^1.7)`, `K_lambda = (10 - 3 lambda)/7`,
`K_H = (1 - |h_H/b|) / (2 l_H/b)^(1/3)`.

In [ ]:
wing_only = lab.subset(L, ["wing_main"], xyz_ref=mrp)
ys = np.linspace(0.02, 0.36, 8)
probe = np.column_stack([np.full_like(ys, 1.48), ys, np.full_like(ys, -0.05)])

def flow_angle(kind, alpha, **kw):
    op = asb.OperatingPoint(atmosphere=lab.atmosphere(0.0), velocity=t7["V"], alpha=alpha)
    s = {"vlm": asb.VortexLatticeMethod, "ll": asb.LiftingLine}[kind](airplane=wing_only, op_point=op, **kw)
    s.run()
    v, fs = s.get_velocity_at_points(probe), op.compute_freestream_velocity_geometry_axes()
    return float(np.mean(np.degrees(np.arctan2(v[:, 2], v[:, 0]) - np.arctan2(fs[2], fs[0]))))

dw = {}
for kind, kw in (("vlm", VLM), ("ll", {})):
    dw[kind] = -(flow_angle(kind, 3.0, **kw) - flow_angle(kind, 1.0, **kw)) / 2.0

secs = L.spec["surfaces"][0]["sections"]
A = b_ref ** 2 / s_ref
lam = secs[-1]["chord"] / secs[0]["chord"]
x_ac_w = lab.slopes(wing_only, cond, "vlm", **VLM)["NP_x"]          # wing aerodynamic centre
ht = L.spec["surfaces"][1]["sections"][0]
x_ac_t = ht["xyz_le"][0] + 0.25 * ht["chord"]
l_H, h_H = x_ac_t - x_ac_w, ht["xyz_le"][2] - secs[0]["xyz_le"][2]
KA, KL, KH = 1 / A - 1 / (1 + A ** 1.7), (10 - 3 * lam) / 7, (1 - abs(h_H / b_ref)) / (2 * l_H / b_ref) ** (1 / 3)
dw["DATCOM 4.4.1"] = 4.44 * (KA * KL * KH) ** 1.19
dw["lifting-line rule 2 CLa_w / (pi A)"] = 2 * lab.slopes(wing_only, cond, "vlm", **VLM)["CLa"] / (math.pi * A)
display(pd.Series(dw, name="de/da").to_frame())
print(f"A {A:.2f}, taper {lam:.3f}, tail arm l_H {l_H:.3f} m, tail height h_H {h_H:+.3f} m, wing AC x {x_ac_w:+.4f} m")

## Step 3. The tail alone

The lift slope of the horizontal tail alone, on its own area, against lifting-line theory (Helmbold) for its aspect
ratio. Then the 2D slope that NeuralFoil gives the tail and wing foils at their Reynolds numbers.

In [ ]:
tail_only = lab.subset(L, ["htail"], xyz_ref=mrp)
S_t = L.airplane.wings[1].area()
A_t = L.airplane.wings[1].span() ** 2 / S_t
tail = {}
for m, kw in (("vlm", VLM), ("ll", {}), ("buildup", {})):
    lo, hi = lab.run_aero(tail_only, cond, 1.0, m, **kw), lab.run_aero(tail_only, cond, 3.0, m, **kw)
    tail[m] = (hi["CL"] - lo["CL"]) / math.radians(2.0) * s_ref / S_t
tail["Helmbold"] = 2 * math.pi * A_t / (2 + math.sqrt(4 + A_t ** 2))
display(pd.Series(tail, name="tail CLa on its own area [1/rad]").to_frame())

foils = {}
for s in L.spec["surfaces"]:
    foils.update(s["foils"])
nu = cond["mu"] / cond["rho"]
al = np.arange(-4.0, 8.01, 0.5)
fig, ax = plt.subplots(figsize=(7, 4))
rows2 = []
for (name, chord), col in zip((("NACA 0012", ht["chord"]), ("EPPLER 395 AIRFOIL", secs[0]["chord"]), ("EPPLER 393 AIRFOIL", 0.3)),
                              ["#2a78d6", "#eb6834", "#1baf7a"]):
    re = t7["V"] * chord / nu
    p = lab.foil_polar(name, foils[name], re, al)
    sl = lambda a0, a1: (np.interp(a1, al, p["CL"]) - np.interp(a0, al, p["CL"])) / math.radians(a1 - a0)
    rows2.append({"foil": name, "Re": re, "slope -1..+1 deg": sl(-1, 1), "slope 2..6 deg": sl(2, 6)})
    ax.plot(al, p["CL"], color=col, label=f"{name}, Re {re:.2g}")
ax.plot(al, 2 * np.pi * np.radians(al), color="#52514e", ls="--", lw=1, label="2 pi alpha")
ax.set(xlabel="alpha [deg]", ylabel="Cl", title="NeuralFoil sections at the T7 speed"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(L.out / "sections_low_re.png", dpi=150); plt.show()
display(pd.DataFrame(rows2).set_index("foil"))
print(f"tail incidence {ht['twist_deg']:.2f} deg: at aircraft alpha 0..5 deg the tail section sees about "
      f"{ht['twist_deg']:.1f}..{ht['twist_deg'] + 5 * (1 - dw['vlm']):.1f} deg, inside the steep band of NACA 0012")

## Step 4. Neutral point by build-up from the checked parts

The textbook neutral point (Etkin & Reid, Dynamics of Flight, chapter 2): `h_n = h_ac + eta V_H (a_t / a_w) (1 - de/da)`, with the
wing AC and slope from the VLM wing, the tail slope from step 3, de/da from step 2, eta = 1. Then the static margin
about the T7 CoG (the MRP) and about the CG of the sheet mass model.

In [ ]:
a_w = lab.slopes(wing_only, cond, "vlm", **VLM)["CLa"]
x_le_mac = lab.mac_geometry(L)["x_le_m"]
h_ac = (x_ac_w - x_le_mac) / c_ref
V_H = S_t * (x_ac_t - x_ac_w) / (s_ref * c_ref)
np_rows = {}
for lbl, a_t, de in (("VLM tail, VLM de/da", tail["vlm"], dw["vlm"]), ("Helmbold tail, DATCOM de/da", tail["Helmbold"], dw["DATCOM 4.4.1"]),
                     ("AeroBuildup tail, no downwash", tail["buildup"], 0.0), ("AeroBuildup tail, VLM de/da", tail["buildup"], dw["vlm"])):
    h_n = h_ac + V_H * a_t / a_w * (1 - de)
    x_np = x_le_mac + h_n * c_ref
    np_rows[lbl] = {"NP_x": x_np, "SM about T7 CoG": (x_np - mrp[0]) / c_ref, "SM about sheet CG": (x_np - cg_now[0]) / c_ref}
np_rows["flow5"] = {"NP_x": M.loc["flow5 (DAVE-ML, T7)", "NP_x"], "SM about T7 CoG": M.loc["flow5 (DAVE-ML, T7)", "SM"],
                    "SM about sheet CG": (M.loc["flow5 (DAVE-ML, T7)", "NP_x"] - cg_now[0]) / c_ref}
for n in ("VLM 16x12", "AeroBuildup full, with boom"):
    np_rows[n + " (step 1)"] = {"NP_x": M.loc[n, "NP_x"], "SM about T7 CoG": M.loc[n, "SM"], "SM about sheet CG": (M.loc[n, "NP_x"] - cg_now[0]) / c_ref}
NP = pd.DataFrame(np_rows).T
display(NP)
print(f"h_ac {h_ac:.3f}, V_H {V_H:.3f}, a_w {a_w:.2f} /rad, MAC LE x {x_le_mac:+.4f} m; T7 CoG x {mrp[0]:.4f} m "
      f"({(mrp[0] - x_le_mac) / c_ref:.1%} MAC), sheet CG x {cg_now[0]:.4f} m ({(cg_now[0] - x_le_mac) / c_ref:.1%} MAC)")

## Step 5. What flow5 implies

flow5 gives CLa and Cma. With the wing AC of the VLM and the tail centre at the tail quarter chord, these fix the
lift slope of each part (as notebook 02 step 2, but with the geometric tail centre, not the elevator derivatives):

    CLa = a_w + a_t S_t/S          Cma = a_w (x_mrp - x_w)/c - a_t S_t/S (x_t - x_mrp)/c

In [ ]:
Am = np.array([[1.0, 1.0], [(mrp[0] - x_ac_w) / c_ref, -(x_ac_t - mrp[0]) / c_ref]])
a_w5, a_t5 = np.linalg.solve(Am, [K["CLa"], K["Cma"]])
x_t_de = mrp[0] - K["Cmde"] / K["CLde"] * c_ref
print(f"flow5 implied: wing {a_w5:.2f} /rad (VLM wing {a_w:.2f}), tail with downwash {a_t5 * s_ref / S_t:.2f} /rad on the tail "
      f"area (VLM {tail['vlm'] * (1 - dw['vlm']):.2f}, Helmbold x (1 - DATCOM de/da) {tail['Helmbold'] * (1 - dw['DATCOM 4.4.1']):.2f})")
print(f"tail centre from the elevator derivatives x {x_t_de:.3f} m; the tail leading edge is at x {ht['xyz_le'][0]:.3f} m")

## Step 6. Lift level of the wing

CL0 and CLa of the wing alone in each method, the fuselage effect in AeroBuildup, and the reference area.

In [ ]:
wl = {}
for n, (m, kw) in {"VLM 16x12": ("vlm", VLM), "LiftingLine": ("ll", {}), "AeroBuildup": ("buildup", {})}.items():
    s = lab.slopes(wing_only, cond, m, **kw)
    wl[n] = {"CLa": s["CLa"], "CL0": s["CL0"], "alpha_L0_deg": s["alpha_L0_deg"]}
wl["flow5 implied (step 5)"] = {"CLa": a_w5, "CL0": np.nan, "alpha_L0_deg": np.nan}
wl["Helmbold, A of S_ref"] = {"CLa": 2 * math.pi * A / (2 + math.sqrt(4 + A ** 2)), "CL0": np.nan, "alpha_L0_deg": np.nan}
display(pd.DataFrame(wl).T)
wb = lab.slopes(lab.subset(L, ["wing_main"], xyz_ref=mrp, bodies=["fuselage"]), cond, "buildup")
print(f"AeroBuildup wing + fuselage: CLa {wb['CLa']:.3f}, CL0 {wb['CL0']:.3f} (AeroBuildup has no wing-body interference)")
print(f"S_ref {s_ref:.4f} m2, AeroSandbox wing area {L.airplane.wings[0].area():.4f} m2, fuselage max width "
      f"{max(x.width for x in L.airplane.fuselages[0].xsecs):.3f} m = {max(x.width for x in L.airplane.fuselages[0].xsecs) / b_ref:.1%} of b")

## Step 7. Drag at the same lift, and the boom

At the flow5 T7 CL (same lift, not same alpha). AeroBuildup by component; the VLM induced drag; flow5 CD0 + K CL^2.

In [ ]:
from scipy.optimize import brentq
CL_t = t7["CL"]
full = lab.subset(L, WT + ["fin"], True, mrp)
a_ab = brentq(lambda a: lab.run_aero(full, cond, a, "buildup")["CL"] - CL_t, -8, 10)
parts = {}
for nm, kw in (("wing", dict(wings=["wing_main"])), ("horizontal tail", dict(wings=["htail"])), ("fin", dict(wings=["fin"])),
               ("fuselage", dict(bodies=["fuselage"])), ("tail boom", dict(bodies=["tail_boom"]))):
    parts[f"AeroBuildup {nm} (alone)"] = lab.run_aero(lab.subset(L, xyz_ref=mrp, **kw), cond, a_ab, "buildup")["CD"]
parts["AeroBuildup total"] = lab.run_aero(full, cond, a_ab, "buildup")["CD"]
vp = lab.subset(L, WT + ["fin"], xyz_ref=mrp)
a_v = brentq(lambda a: lab.run_aero(vp, cond, a, "vlm", **VLM)["CL"] - CL_t, -8, 10)
parts["VLM induced"] = lab.run_aero(vp, cond, a_v, "vlm", **VLM)["CD"]
parts["flow5 CD0 + K CL^2"] = K["CD0"] + K["K"] * CL_t ** 2
D = pd.Series(parts, name="CD").to_frame()
qSV = cond["q"] * s_ref * cond["V"]
D["power at T7 speed [W]"] = D["CD"] * qSV
display(D)
print(f"alpha for CL {CL_t:.3f}: AeroBuildup {a_ab:.2f} deg, VLM {a_v:.2f} deg, flow5 {t7['alpha_deg']:.2f} deg")
nb = M.loc["AeroBuildup full, no boom"]; wb_ = M.loc["AeroBuildup full, with boom"]
print(f"boom: dCD {parts['AeroBuildup tail boom (alone)']:.5f} ({parts['AeroBuildup tail boom (alone)'] * qSV:.2f} W), "
      f"dCLa {wb_['CLa'] - nb['CLa']:+.4f}, dCma {wb_['Cma'] - nb['Cma']:+.4f} /rad")

## Findings

The cell prints the numbers that the report quotes, so they stay true when the source tree changes.

In [ ]:
f = M
print("1. Tail slope on its own area [1/rad]: VLM", round(tail["vlm"], 2), "Helmbold", round(tail["Helmbold"], 2),
      "LiftingLine", round(tail["ll"], 2), "AeroBuildup", round(tail["buildup"], 2))
print("   NeuralFoil NACA 0012 slope -1..+1 deg:", round(rows2[0]["slope -1..+1 deg"], 2), "at Re", f"{rows2[0]['Re']:.3g}")
print("2. de/da: VLM", round(dw["vlm"], 3), "LiftingLine", round(dw["ll"], 3), "DATCOM", round(dw["DATCOM 4.4.1"], 3), "AeroBuildup 0")
print("3. NP x [m]: flow5", round(NP.loc["flow5", "NP_x"], 3), "| VLM", round(f.loc["VLM 16x12", "NP_x"], 3),
      "| build-up VLM parts", round(NP.loc["VLM tail, VLM de/da", "NP_x"], 3), "| Helmbold+DATCOM", round(NP.loc["Helmbold tail, DATCOM de/da", "NP_x"], 3),
      "| AeroBuildup", round(f.loc["AeroBuildup full, with boom", "NP_x"], 3))
print("   SM about the sheet CG: VLM", round(NP.loc["VLM 16x12 (step 1)", "SM about sheet CG"], 3), "Helmbold+DATCOM",
      round(NP.loc["Helmbold tail, DATCOM de/da", "SM about sheet CG"], 3), "flow5", round(NP.loc["flow5", "SM about sheet CG"], 3))
print("4. Wing CLa: VLM", round(wl["VLM 16x12"]["CLa"], 2), "flow5 implied", round(a_w5, 2), "| CL0 total: flow5", round(K["CL0"], 3),
      "VLM", round(f.loc["VLM 16x12", "CL0"], 3), "AeroBuildup", round(f.loc["AeroBuildup full, with boom", "CL0"], 3))
print("5. Drag at CL", round(CL_t, 3), ": AeroBuildup", round(parts["AeroBuildup total"], 4), "flow5", round(parts["flow5 CD0 + K CL^2"], 4),
      "| boom", round(parts["AeroBuildup tail boom (alone)"], 5))